In [ ]:
from Bio import SeqIO
from datasets import Dataset, DatasetDict
from transformers import EsmTokenizer

model_name = 'facebook/esm2_t6_8M_UR50D'
tokenizer = EsmTokenizer.from_pretrained(model_name)

def fasta_to_ds(fasta_path):
    """
    Reads a fasta file and transcribes its
    contents into a HuggingFace Dataframe
    """
    data = {'id': [], 'description': [], 'sequence': [], 'length': []}

    with open(fasta_path) as handle:
        for rec in SeqIO.parse(handle, "fasta"):
            seq_str = str(rec.seq)
            data['id'].append(rec.id)
            data['description'].append(rec.description)
            data['sequence'].append(seq_str)
            data['length'].append(len(seq_str))

    return Dataset.from_dict(data)

def preprocess_function(examples):
    """Returns the tokenization of a batch of examples"""
    return tokenizer(examples['sequence'], truncation=True, max_length=1024)

In [32]:
dataset_folder = '../../data/training/maskedlm/'

def preprocess_dataset(filename):
    """
    Preprocesses a FASTA file into an ESM2 token dataset
    divided into splits and pushes it to the HuggingFace Hub
    """
    # Dump FASTA contents into dataset
    dataset = fasta_to_ds(dataset_folder + filename + '.fasta')
    # Tokenize the dataset's sequences
    dataset = dataset.map(function=preprocess_function, batched=True, remove_columns=dataset.column_names)

    # Split the dataset 
    train_eval = dataset.train_test_split(test_size=0.2, shuffle=True)
    # Format into a dataset divided into splits
    dataset = DatasetDict({'train': train_eval['train'], 'eval': train_eval['test']})

    # Format name and save to disk and Hub
    dataset_name = filename + '-tokenized'
    dataset.save_to_disk(dataset_folder + dataset_name)
    dataset.push_to_hub(dataset_name, private=False)

In [33]:
fasta_datasets = ['fnbp-70K', 'mscramm-100K', 'mscramm-227K', 'cwa-500K', 'cwa-1M']

for filename in fasta_datasets:
    print('Processing:', filename)
    preprocess_dataset(filename)    

Processing: fnbp-70K


Map:   0%|          | 0/72106 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/57684 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14422 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/58 [00:00<?, ?ba/s]

... (35 outputs left out)

Creating parquet from Arrow format:   0%|          | 0/209 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/209 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/209 [00:00<?, ?ba/s]